# 4xL4 topology + NCCL all-reduce check (E025)

PCIe link info, P2P status, all-reduce timing (default / NCCL_P2P_LEVEL=NODE / SYS) and correctness. No model. Hard 11-minute budget.

In [ ]:
# E025: 4xL4 topology + NCCL all-reduce check. Hard budget: everything stops 9 min after this cell starts.
import os, subprocess, time, glob, sys
T0 = time.time(); DEADLINE = T0 + 9 * 60
LIB = "/usr/local/nvidia/lib64"
if os.path.exists(LIB + "/libcuda.so"):  # host driver libraries (as the solver's driver_env)
    os.environ["LD_LIBRARY_PATH"] = LIB + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_PRELOAD"] = LIB + "/libcuda.so"
os.environ["PATH"] = "/usr/local/nvidia/bin:" + os.environ.get("PATH", "")
os.makedirs("/kaggle/working/p2p", exist_ok=True)
open("/kaggle/working/p2p/allreduce_bench.py", "w").write('"""NCCL all-reduce timing across the 4 GPUs for tensor sizes TP4 inference uses per all-reduce\n(bf16 [tokens, 5120]): decode steps (24-48 tokens) and prefill chunks (2048-8192 tokens).\nA TP4 forward of Qwen3.8-27B does 2 all-reduces per layer x 64 layers = 128 per step.\nusage: torchrun --nproc_per_node 4 allreduce_bench.py"""\nimport os, time, torch, torch.distributed as dist\n\ndist.init_process_group("nccl")\nr = dist.get_rank()\ntorch.cuda.set_device(r)\nfor tokens in (24, 48, 512, 2048, 4096, 8192):\n    x = torch.randn(tokens, 5120, dtype=torch.bfloat16, device="cuda")\n    for _ in range(5):\n        dist.all_reduce(x)\n    torch.cuda.synchronize()\n    n = 50 if tokens <= 2048 else 20\n    t0 = time.time()\n    for _ in range(n):\n        dist.all_reduce(x)\n    torch.cuda.synchronize()\n    dt = (time.time() - t0) / n\n    if r == 0:\n        mb = x.numel() * 2 / 1e6\n        print(f"tokens {tokens:5d}  {mb:7.1f} MB  {dt * 1e3:7.2f} ms per all-reduce  "\n              f"bus {2 * 3 / 4 * mb / 1e3 / dt:6.1f} GB/s  x128 per step = {128 * dt:6.3f} s", flush=True)\ndist.destroy_process_group()\n')
open("/kaggle/working/p2p/allreduce_check.py", "w").write('"""Correctness of NCCL all-reduce across the 4 GPUs (run with and without NCCL_P2P_LEVEL=SYS):\nrandom bf16/fp32 tensors of several sizes, all-reduce result compared with the sum computed from gathered inputs.\nusage: torchrun --nproc_per_node 4 allreduce_check.py"""\nimport torch, torch.distributed as dist\n\ndist.init_process_group("nccl")\nr = dist.get_rank()\nw = dist.get_world_size()\ntorch.cuda.set_device(r)\nbad = 0\nfor it in range(40):\n    for tokens in (1, 24, 48, 511, 2048, 8192):\n        g = torch.Generator(device="cuda").manual_seed(1000 * it + tokens * 7 + r)\n        x = torch.randn(tokens, 5120, dtype=torch.float32, device="cuda", generator=g)\n        parts = [torch.empty_like(x) for _ in range(w)]\n        dist.all_gather(parts, x)\n        ref = torch.stack(parts).sum(0)\n        y = x.clone()\n        dist.all_reduce(y)\n        err = (y - ref).abs().max().item()\n        if err > 1e-3:\n            bad += 1\n            if bad <= 5:\n                print(f"rank {r} it {it} tokens {tokens}: max abs err {err}", flush=True)\nt = torch.tensor([bad], device="cuda")\ndist.all_reduce(t)\nif r == 0:\n    print(f"all-reduce check: {int(t.item())} mismatching tensors out of {40 * 6 * w} rank-checks", flush=True)\ndist.destroy_process_group()\n')
def sh(cmd, limit=120, env=None):
    left = DEADLINE - time.time()
    if left < 20:
        print("SKIP (deadline):", cmd, flush=True); return ""
    e = dict(os.environ); e.update(env or {})
    print(f"$ [{time.time() - T0:5.0f}s] {cmd}", flush=True)
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=e, timeout=min(limit, left - 10))
        out = (r.stdout or "") + (r.stderr or "")
    except subprocess.TimeoutExpired:
        out = "TIMEOUT"
    print(out[-5000:], flush=True)
    return out
sh("nvidia-smi --query-gpu=name,pci.bus_id,driver_version,pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max --format=csv")
sh("nvidia-smi topo -m")
sh("nvidia-smi topo -p2p r")
sh("nvidia-smi topo -p2p w")
sh("(lspci -tv 2>/dev/null || ls /sys/bus/pci/devices) | head -40")


In [ ]:
# CUDA torch (+ NCCL) from the attached vLLM 0.23 wheel source; the base image torch is a CPU build.
whl = sorted(glob.glob("/kaggle/input/**/vllm-0.23.0*.whl", recursive=True))
WD = os.path.dirname(whl[0]) if whl else None
print("wheel dir:", WD, flush=True)
if WD:
    out = sh(f"uv pip install --system --python {sys.executable} --no-index --find-links {WD} 'torch==2.11.0+cu130'", limit=420)
    if "TIMEOUT" in out or "error" in out.lower()[-500:]:
        sh(f"{sys.executable} -m pip install --no-index --find-links {WD} 'torch==2.11.0+cu130'", limit=300)
sh(f"{sys.executable} -c 'import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.nccl.version())'")


In [ ]:
for name, env in [("default", {}), ("P2P_LEVEL=NODE", {"NCCL_P2P_LEVEL": "NODE"}), ("P2P_LEVEL=SYS", {"NCCL_P2P_LEVEL": "SYS"})]:
    print("=" * 20, name, flush=True)
    sh("NCCL_DEBUG=INFO torchrun --nproc_per_node 4 /kaggle/working/p2p/allreduce_bench.py 2>&1 | grep -E '^tokens|via (P2P|SHM|NET)' | sort | uniq -c | sort -rn | head -12", 100, env)
print("=" * 20, "correctness, default and P2P_LEVEL=SYS", flush=True)
sh("torchrun --nproc_per_node 4 /kaggle/working/p2p/allreduce_check.py 2>&1 | grep -E 'all-reduce check|err' | head -4", 90)
sh("torchrun --nproc_per_node 4 /kaggle/working/p2p/allreduce_check.py 2>&1 | grep -E 'all-reduce check|err' | head -4", 90, {"NCCL_P2P_LEVEL": "SYS"})
print(f"total {time.time() - T0:.0f}s", flush=True)
